In [ ]:
!pip install numpy

In [ ]:
!pip install pandas gensim

In [ ]:
import pandas as pd
import numpy as np
import nltk
import string
# import gensim

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
# from gensim.models import Word2Vec
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

In [ ]:
eng_stopw = stopwords.words('english')
lemmatizer = WordNetLemmatizer()
punctuation = string.punctuation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
dataset = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Dataset_AOL_NLP/train (2).csv', delimiter=';')

In [ ]:
print(dataset.isnull().sum())

In [ ]:
dataset['label'].fillna(dataset['label'].mode()[0] , inplace = True)

Terdapat 1 kolom label yang kosong, maka akan kita isi menggunakan mode

In [ ]:
print(dataset.isnull().sum())

In [ ]:
frekuensi_label = dataset["label"].value_counts()
print(frekuensi_label)

In [ ]:
dataset.info()

In [ ]:
dataset.drop(columns=['Unnamed: 0'] , inplace=True)

In [ ]:
dataset.info()

In [ ]:
duplicates = dataset.duplicated()

In [ ]:
print(f"Jumlah duplikat: {duplicates.sum()}")
dataset_cleaned = dataset.drop_duplicates()
dataset_cleaned = dataset_cleaned.reset_index(drop=True)
print(dataset_cleaned.index)

In [ ]:
x = dataset['title']
y = dataset['text']
print(x.head())
print('======================================================')
print(y.head())

In [ ]:
def preprocessing_sentence(text):
  words = word_tokenize(text.lower())

  words = [w for w in words if w not in eng_stopw and w not in punctuation and w.isalpha()]
  words = [lemmatizer.lemmatize(word) for word in words]

  return words

In [ ]:
clean_text = dataset_cleaned['text'].apply(preprocessing_sentence)
clean_title = dataset_cleaned['title'].apply(preprocessing_sentence)

In [ ]:
print(clean_text)
print(clean_title)

In [ ]:
X = pd.DataFrame({'title': clean_title, 'text': clean_text})
y = dataset_cleaned['label']

In [ ]:
X['combined_text'] = X['title'] + X['text']

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [ ]:
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [ ]:
X_train['combined_text'] = X_train['title'] + X_train['text']
X_val['combined_text'] = X_val['title'] + X_val['text']
X_test['combined_text'] = X_test['title'] + X_test['text']

# Model + Vectorizer -> BERT

In [ ]:
import transformers
import torch

In [ ]:
from transformers import BertTokenizer
from transformers import BertForSequenceClassification

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

Pastikan semua data type adalah string

In [ ]:
X_train['combined_text'] = X_train['combined_text'].astype(str)
X_val['combined_text'] = X_val['combined_text'].astype(str)
X_test['combined_text'] = X_test['combined_text'].astype(str)

In [ ]:
train_encodings = tokenizer(list(X_train['combined_text']), truncation=True, padding=True, max_length=512, return_tensors='pt')
val_encodings = tokenizer(list(X_val['combined_text']), truncation=True, padding=True, max_length=512, return_tensors='pt')
test_encodings = tokenizer(list(X_test['combined_text']), truncation=True, padding=True, max_length=512, return_tensors='pt')

In [ ]:
class HoaxDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

    def __len__(self):
        return len(self.labels)

train_dataset = HoaxDataset(train_encodings, y_train)
val_dataset = HoaxDataset(val_encodings, y_val)
test_dataset = HoaxDataset(test_encodings, y_test)

In [ ]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

In [ ]:
for name, param in model.bert.named_parameters():
    if not any(layer in name for layer in ['encoder.layer.10', 'encoder.layer.11']):
        param.requires_grad = False

In [ ]:
from transformers import Trainer, TrainingArguments

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
    run_name='bert-hoax-classification-run-2'
)
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
model.save_pretrained('/content/drive/MyDrive/model_hoax_trained_final')
tokenizer.save_pretrained('/content/drive/MyDrive/model_hoax_trained_final')

In [ ]:
save_directory = "/content/drive/MyDrive/Colab Notebooks/Dataset_AOL_NLP/model_hoax_BERT_final"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

In [ ]:
from transformers import BertForSequenceClassification, BertTokenizerFast

model = BertForSequenceClassification.from_pretrained(save_directory)
tokenizer = BertTokenizerFast.from_pretrained(save_directory)

In [ ]:
trainer = Trainer(model=model)
predictions = trainer.predict(test_dataset)

In [ ]:
y_pred_logits = predictions.predictions
y_true = predictions.label_ids

y_pred = np.argmax(y_pred_logits, axis=1)


acc = accuracy_score(y_true, y_pred)
print(f"Akurasi: {acc:.4f}")

In [ ]:
import os
os.listdir(save_directory)